## Load packages

In [6]:
%load_ext autoreload
%autoreload 2
# Do functions to retrieve embedding for WES (they will be of shape n x genes x channels)
# Check why Jhon did not use oncogenic and why he did not apply dimensionality reduction
# Check for NAs


# Load packages and classes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle as pkl
import tiffslide
import seaborn as sns
import gget, os
import tifffile
import zarr
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from scipy.stats import ttest_1samp
from statsmodels.stats.multitest import multipletests
from torch.utils.data import Dataset, DataLoader
# MosaicDataset and BruceDataset classes allow loading and visualisation of the different data sources
from gbmhackathon import MosaicDataset 
from gbmhackathon.s3_loader import load_s3, write_s3

from sklearn.linear_model import Lasso
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_validate, train_test_split
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from umap import UMAP

from foundation.wes import pipeline_wes

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
MosaicDataset

DataCenter(name='mosaic', sources={'clinical': DataSource(files={'data dictionary': DataFile(name=PosixPath('03bb30aa-16ed-4b89-913e-fe009db2aabd/Clinical/GBM_HK_data_dictionary.csv'), loader=<gbmhackathon.data.io.loaders.CSVDataLoader object at 0x7fb9288a1330>), 'original clinical': DataFile(name=PosixPath('03bb30aa-16ed-4b89-913e-fe009db2aabd/Clinical/GBM_HK_sample_and_clinical_data.csv'), loader=<gbmhackathon.data.io.loaders.CSVDataLoader object at 0x7fb91e7c6b30>), 'processed gbm clinical': DataFile(name=PosixPath('03bb30aa-16ed-4b89-913e-fe009db2aabd/Clinical/GBM_HK_sample_and_clinical_data.csv'), loader=<gbmhackathon.data.io.loaders.CSVDataLoader object at 0x7fb91e6289a0>), 'treatments': DataFile(name=PosixPath('03bb30aa-16ed-4b89-913e-fe009db2aabd/Clinical/GBM_HK_multi_entry_treatments.csv'), loader=<gbmhackathon.data.io.loaders.CSVDataLoader object at 0x7fb91e632a70>), 'key events clinical': DataFile(name=PosixPath('03bb30aa-16ed-4b89-913e-fe009db2aabd/Clinical/GBM_HK_multi_ent

## Remove duplicated rows and columns

In [3]:
def remove_duplicates(df):
    df = df.loc[:,~df.columns.duplicated()] #remove duplicated columns
    df = df.loc[~df.index.duplicated(),:] #remove duplicated rows
    return df

In [4]:
source_dict_mosaic = MosaicDataset.load_tabular()

onc = remove_duplicates(source_dict_mosaic["wes"]["WES CNV oncogenic"])
dele = remove_duplicates(source_dict_mosaic["wes"]["WES CNV deletion"])
amp = remove_duplicates(source_dict_mosaic["wes"]["WES CNV amplification"])
mut = remove_duplicates(source_dict_mosaic["wes"]["WES mutations"])

In [5]:
for df in [onc, dele, amp, mut]:
    print(df.sum(axis=0))

gene_name
TSPAN6             0
TNMD               0
DPM1               0
SCYL3              0
FIRRM              0
                  ..
POLGARF            0
LY6S               0
TMEM276-ZFTRAF1    0
TMEM276            0
DUSP13A            0
Length: 19429, dtype: int64
gene_name
TSPAN6              2
TNMD                2
DPM1                0
SCYL3               0
FIRRM               1
                   ..
POLGARF             0
LY6S                0
TMEM276-ZFTRAF1     0
TMEM276             0
DUSP13A            28
Length: 19429, dtype: int64
gene_name
TSPAN6             2
TNMD               2
DPM1               0
SCYL3              0
FIRRM              0
                  ..
POLGARF            5
LY6S               0
TMEM276-ZFTRAF1    0
TMEM276            0
DUSP13A            0
Length: 19429, dtype: int64
gene_name
CREBBP    0
CD79B     0
BTK       1
BRCA1     1
FAS       0
         ..
CUX1      0
TAF15     2
KMT2B     1
ZNF658    0
H3C2      0
Length: 498, dtype: int64


## Filter out columns where all values are False (no useable information)

In [6]:
def retrieve_useful_cols(df):
    query = np.sum(df, axis=0)
    return list(set(query[query > 0].index))

In [7]:
useful_cols = {}
for key, df in {'onc':onc, 'del':dele, 'amp':amp, 'mut':mut}.items():
    useful_cols[key] = retrieve_useful_cols(df)

In [8]:
total = []
col_set = set()
for key in useful_cols.keys():
    print(len(useful_cols[key]))
    total += useful_cols[key]
    col_set = col_set.union(set(useful_cols[key]))
uniques, counts = np.unique(total, return_counts=True)
counts_dico = dict(zip(list(uniques), list(counts)))
duplicated = [col for col in counts_dico.keys() if counts_dico[col] > 1]
print(f"Total number of useful columns from all dataframes {len(total)}\nNumber of unique columns {len(col_set)}.\nThere are {len(duplicated)} duplicated columns")

257
10729
13790
209
Total number of useful columns from all dataframes 24985
Number of unique columns 17328.
There are 7354 duplicated columns


## Check if WES modalities encode different informations about exome
**I am currently making the asusmption that duplicated columns correspond to different biological signals if they are in different dataframes, this means that each duplicated gene column is distinct. This can be verified quickly : If columns which have the same name, are ordered in the same way regarding sample ids and have different values, this means that both correspond to distinct informations. If we can find only one example where this is the case this should confirm that same gene columns correspond to different informations across WES modalities.**


In [9]:
def check_identical(df_dict):
    # Identify columns present in all dataframes
    common_columns = set.intersection(*(set(df.columns) for df in df_dict.values()))

    # Determine which of these common columns have identical values across all dataframes
    identical_columns = set()
    for col in common_columns:
        # Extract the column from each dataframe and check if all are identical
        col_values = [df[col] for df in df_dict.values()]
        if all(col.equals(col_values[0]) for col in col_values):
            pass
        else:
            print(col_values)
            return "There are columns for the same genes which are different"

In [10]:
raw_onc = remove_duplicates(source_dict_mosaic["wes"]["WES CNV oncogenic"])[useful_cols['onc']]
raw_dele = remove_duplicates(source_dict_mosaic["wes"]["WES CNV deletion"])[useful_cols['del']]
raw_amp = remove_duplicates(source_dict_mosaic["wes"]["WES CNV amplification"])[useful_cols['amp']]
raw_mut = remove_duplicates(source_dict_mosaic["wes"]["WES mutations"])[useful_cols['mut']]

In [11]:
for df1 in [raw_onc, raw_dele, raw_amp, raw_mut]:
    for df2 in [raw_onc, raw_dele, raw_amp, raw_mut]:
        print(list(df1.index) == list(df2.index))

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


**All indices are exactly in the same order. This means that we can procede with checking which columns are identical or not.**

In [12]:
dfs = {
    "onc": raw_onc,
    "del": raw_dele,
    "amp": raw_amp,
    "mut": raw_mut
}

check_identical(dfs)

[sample_id
HK_G_001a    False
HK_G_002a    False
HK_G_003a    False
HK_G_004a    False
HK_G_005a    False
             ...  
HK_G_111b    False
HK_G_112a    False
HK_G_113b    False
HK_G_114a    False
HK_G_115b    False
Name: USP13, Length: 107, dtype: bool, sample_id
HK_G_001a    False
HK_G_002a    False
HK_G_003a    False
HK_G_004a    False
HK_G_005a    False
             ...  
HK_G_111b    False
HK_G_112a    False
HK_G_113b    False
HK_G_114a    False
HK_G_115b    False
Name: USP13, Length: 107, dtype: bool, sample_id
HK_G_001a    False
HK_G_002a     True
HK_G_003a    False
HK_G_004a    False
HK_G_005a    False
             ...  
HK_G_111b    False
HK_G_112a     True
HK_G_113b    False
HK_G_114a    False
HK_G_115b     True
Name: USP13, Length: 107, dtype: bool, sample_id
HK_G_001a    False
HK_G_002a    False
HK_G_003a    False
HK_G_004a    False
HK_G_005a    False
             ...  
HK_G_111b    False
HK_G_112a    False
HK_G_113b    False
HK_G_114a    False
HK_G_115b    False
Name: 

'There are columns for the same genes which are different'

**This confirms that a priori, each WES modality corresponds to a different information**

## Add identifier to column names

In [13]:
def update_name(col_name, df_name):
    if col_name in duplicated:
        return f"{col_name}.{df_name}"
    return col_name

In [14]:
#Update name with identifier
onc_kept_cols = [update_name(col, 'onc') for col in useful_cols['onc']]
# Retrieve kept columns
onc = onc[useful_cols['onc']]
# Update column names accordingly
onc.columns = onc_kept_cols

del_kept_cols = [update_name(col, 'del') for col in useful_cols['del']]
# Retrieve kept columns
dele = dele[useful_cols['del']]
# Update column names accordingly
dele.columns = del_kept_cols

amp_kept_cols = [update_name(col, 'amp') for col in useful_cols['amp']]
# Retrieve kept columns
amp = amp[useful_cols['amp']]
# Update column names accordingly
amp.columns = amp_kept_cols

mut_kept_cols = [update_name(col, 'mut') for col in useful_cols['mut']]
# Retrieve kept columns
mut = mut[useful_cols['mut']]
# Update column names accordingly
mut.columns = mut_kept_cols

KeyboardInterrupt: 

In [ ]:
rdy_onc = onc[onc_kept_cols]
rdy_del = dele[del_kept_cols]
rdy_amp = amp[amp_kept_cols]
rdy_mut = mut[mut_kept_cols]

X_df = pd.concat([rdy_onc, rdy_del, rdy_amp, rdy_mut], axis=1)

## Feature Selection
**Because we have a huge amount of genes, let's select a subset of them to keep. To do that we will use a Lasso Regressor. with respect to each possible target objective. We will then keep all significantly important features across targets.**

In [ ]:
lasso = Lasso()
logreg = LogisticRegression(penalty="l1", solver='saga', class_weight='balanced')
rfr = RandomForestRegressor(max_features='log2')
rfc = RandomForestClassifier(max_features='sqrt', class_weight='balanced')

In [ ]:
BUCKET_PROJECT = "ABSTRA_PROJECT_STORAGE_BUCKET"

def fetch_path(env_var_name):
    return os.path.expandvars(f"${env_var_name}")

In [ ]:
S3_PATH_CLINICAL_EMB = fetch_path(BUCKET_PROJECT) + "embedding_V1/2025-03-30_14-23_clinical_emb_V1.pkl"
clinical_dict = load_s3(S3_PATH_CLINICAL_EMB)

In [ ]:
Y = clinical_dict['dataset']['Y']
id2row = clinical_dict['dataset']['id2row']
targets = clinical_dict['dataset']['targets']
Y_df = pd.DataFrame(Y.numpy(), columns=targets, index=list(id2row.keys()))
Y_df = Y_df.loc[X_df.index]

In [ ]:
def evaluate(estimator, x_df, y_df, targets, mode='clf', cv=5, verbose=True):
    if mode == 'clf':
        scoring = 'roc_auc'
    else:
        scoring = 'neg_mean_absolute_error'
    RESULTS = {}
    for target in targets:
        print(f"\nEvaluating for {target}..")
        results = cross_validate(estimator, X_df, y_df[target], cv=cv, n_jobs=-1, scoring=scoring, return_estimator=True)
        estimators = results['estimator']
        scores = results['test_score']
        scores = [-s for s in scores] if mode != 'clf' else scores
        if verbose:
            print(f"{target} MAX: {np.max(y_df[target])}, MIN: {np.min(y_df[target])}")
            print(scores)
        

        RESULTS[target] = {"estimators":estimators, "scores":scores}
    return RESULTS

def cross_target_mean(dico):
    score = []
    for target in dico.keys():
        score += list(dico[target]['scores'])
    return np.mean(score)

In [ ]:
lasso_RESULTS = evaluate(lasso, X_df, Y_df, targets[:-1], mode='reg', cv=5)

In [ ]:
rfr_RESULTS = evaluate(rfr, X_df, Y_df, targets[:-1], mode='reg', cv=5)

In [ ]:
print(f"Cross target mean for Lasso: {cross_target_mean(lasso_RESULTS)}")
print(f"\nCross target mean for RFR: {cross_target_mean(rfr_RESULTS)}")

**So across all regression targets, Lasso yields lowest average error.**

In [ ]:
logreg_RESULTS = evaluate(logreg, X_df, Y_df, targets[-1:], mode='clf', cv=5)

In [ ]:
rfc_RESULTS = evaluate(rfc, X_df, Y_df, targets[-1:], mode='clf', cv=5)

In [ ]:
print(f"Cross target mean for logisticRegression: {cross_target_mean(logreg_RESULTS)}")
print(f"\nCross target mean for RFC: {cross_target_mean(rfc_RESULTS)}")

**So on the classification task, LogisticRegression yields highest average AUC score. So we will look at feature importances from Lasso and Logistic Regression.**

In [ ]:
def get_feature_imp(estimators, type):
    out_dict = {}
    for target in estimators.keys():
        for i, estimator in enumerate(estimators[target]['estimators']):
            features = estimator.feature_names_in_
            if type == 'tree':
                imp = estimator.feature_importances_.reshape(1,-1)
            else:
                imp = estimator.coef_.reshape(1,-1)
            for i, col in enumerate(features):
                if col in out_dict.keys():
                    out_dict[col].append(imp[0,i])
                else:
                    out_dict[col] = [imp[0,i]]
    return out_dict

In [ ]:
cross_targets_rfr_imp_df = pd.DataFrame(get_feature_imp(rfr_RESULTS, 'tree'))
rfr_imp_means = cross_targets_rfr_imp_df.mean(axis=0)
p = np.percentile(rfr_imp_means, 95)
cross_targets_rfr_imp_df.loc[:,rfr_imp_means > p].columns

### Identify Significantly Important features

In [ ]:
def get_signif_features(results, mode, alpha=0.05):
    print("Retrieving Importances..")
    cross_targets_imp_df = pd.DataFrame(get_feature_imp(results, mode))
    imp_means = cross_targets_imp_df.mean(axis=0)

    print("Computing raw p-values for each feature..")
    # Compute p-values from one-sample t-tests for each feature
    p_values = {}
    signif_features_noadj = []
    for feature in tqdm(cross_targets_imp_df.columns):
        # ttest_1samp tests whether the mean of the sample equals popmean (here, 0)
        t_stat, p_val = ttest_1samp(cross_targets_imp_df[feature], popmean=0)
        p_values[feature] = p_val
        if p_val < alpha:
            signif_features_noadj.append(feature)

    print("Significant features (No correction):")
    print(len(signif_features_noadj))

    # Convert p-values to an array for correction
    features = list(p_values.keys())
    pvals = np.array([p_values[feat] for feat in tqdm(features)])

    print("Applying Bonferroni correction..")
    # Apply Bonferroni correction
    reject_bonf, pvals_bonf, _, _ = multipletests(pvals, alpha=alpha, method='bonferroni')
    signif_features_bonf = [features[i] for i, rej in tqdm(enumerate(reject_bonf)) if rej]
    
    print("Significant features (Bonferroni correction):")
    print(len(signif_features_bonf))

    print("Applying Benjamin-Hochberg correction..")
    # Alternatively, apply Benjamini-Hochberg (FDR) correction
    reject_fdr, pvals_fdr, _, _ = multipletests(pvals, alpha=alpha, method='fdr_bh')
    signif_features_fdr = [features[i] for i, rej in tqdm(enumerate(reject_fdr)) if rej]
    
    print("\nSignificant features (FDR correction):")
    print(len(signif_features_fdr))
    return {'none':signif_features_noadj, 'bonf':signif_features_bonf, 'fdr':signif_features_fdr}

In [ ]:
signif_lasso = get_signif_features(lasso_RESULTS, 'coef')

**Cannot identify significant features using Lasso coefficients. Let's see for RandomForestRegressor impurity based importance.**

In [ ]:
signif_rfr = get_signif_features(rfr_RESULTS, 'tree')

**After multi hypothesis correction no feature is identied as significant. Thus we will only use features identified without adjustment because we still need to choose features. Now let's look for classification task.**

In [ ]:
signif_logreg = get_signif_features(logreg_RESULTS, 'coef')

In [ ]:
signif_rfc = get_signif_features(rfc_RESULTS, 'tree')

In [ ]:
selected_all = list(set(signif_lasso['none'] + signif_rfr['none'] + signif_logreg['none'] + signif_rfc['none']))
print(f"Aross all selection approaches we obtain : {len(selected_all)} unique relevant features")

In [ ]:
X_final_df = X_df[selected_all]
X_final_df.head()

In [ ]:
count = {'unique':0}
for col in X_final_df.columns:
    if '.' in col:
        suffix = col[-3:]
        if suffix in count.keys():
            count[suffix] += 1
        else:
            count[suffix] = 1
    else:
        count['unique'] += 1
            
count

In [ ]:
print(list(X_final_df.columns))

Let's see if applying dimensionality reduction helps

In [ ]:
pca_emb = PCA().fit_transform(X_final_df)
tsne_emb = TSNE(n_components=3, random_state=6262).fit_transform(X_final_df) # TSNE requires n < 3
umap_emb = UMAP(n_components=50).fit_transform(X_final_df)

In [ ]:
for n in [5, 10, 20, 50]:
    print(f"**** FOR n_components = {n} ****")
    print(f"\nCross target mean for Lasso: {cross_target_mean(evaluate(lasso, pca_emb[:,:n], Y_df, targets[:-1], 'reg', verbose=False))}")

In [ ]:
for n in [5, 10, 20, 50]:
    print(f"**** FOR n_components = {n} ****")
    print(f"\nCross target mean for Lasso: {cross_target_mean(evaluate(lasso, tsne_emb[:,:n], Y_df, targets[:-1], 'reg', verbose=False))}")

In [ ]:
for n in [5, 10, 20, 50]:
    print(f"**** FOR n_components = {n} ****")
    print(f"\nCross target mean for Lasso: {cross_target_mean(evaluate(lasso, umap_emb[:,:n], Y_df, targets[:-1], 'reg', verbose=False))}")

**The conclusion is that applying dimensionality reduction techniques does not seem to be helpful regarding overal regression performance**

In [ ]:
for n in [5, 10, 20, 50]:
    print(f"**** FOR n_components = {n} ****")
    print(f"\nCross target mean for LogisticRegression: {cross_target_mean(evaluate(logreg, pca_emb[:,:n], Y_df, [targets[-1]], verbose=False))}")

In [ ]:
for n in [5, 10, 20, 50]:
    print(f"**** FOR n_components = {n} ****")
    print(f"\nCross target mean for LogisticRegression: {cross_target_mean(evaluate(logreg, tsne_emb[:,:n], Y_df, [targets[-1]], verbose=False))}")

In [ ]:
for n in [5, 10, 20, 50]:
    print(f"**** FOR n_components = {n} ****")
    print(f"\nCross target mean for LogisticRegression: {cross_target_mean(evaluate(logreg, umap_emb[:,:n], Y_df, [targets[-1]], verbose=False))}")

**Same conclusion for classification**

In [ ]:
np.sum(np.sum(pd.isna(X_final_df)))

**There are no NAs (already known since we were able to train models, however it is a good sanity check as this is the df we will end up using)**

## What is needed for embedding script
1) Load data
2) Apply preprocessing steps
3) Filter out the dataframe using the final selected list in the cell above
4) Turn into tensor

In [ ]:
os.getcwd()

In [ ]:
# with open("wes_selected_features.pkl", "wb") as f: 
#    pkl.dump(list(X_final_df.columns), f)

## (Script has been made)

In [7]:
wes_data = pipeline_wes()

In [9]:
write_s3(wes_data, 'wes_emb_V1', 'embedding_V1')

Object saved at s3://abstra-project-storage-lttemftb/1b75dc89-ad27-4a65-9e7f-877d1b4f36fc/embedding_V1/2025-04-05_13-40_wes_emb_V1.pkl
